# Combined ICU Mortality Model Training and CSV Export

This notebook combines:

- Logistic Regression tuning
- XGBoost ROC-AUC tuning, using the same grid/search setup as the previous `xgboost.ipynb`
- Optional XGBoost Brier-score tuning
- Feed-forward MLP tuning using an internal validation split

This notebook **does not create figures**.  
It saves model outputs, performance metrics, ROC/PR/calibration curve data, and optional explanation tables as CSV files.

Main rule:

**Do not use the external test set for tuning.**  
The external test set is evaluated only after each model has already been selected.

In [1]:
# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import copy
import json
import random
import warnings
import platform
import sys
import time

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

import joblib

try:
    import xgboost as xgb
except ImportError as e:
    raise ImportError("xgboost is required for this notebook. Install it with `pip install xgboost`.") from e

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
except ImportError as e:
    raise ImportError("PyTorch is required for the MLP section. Install it before running this notebook.") from e

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("XGBoost:", xgb.__version__)
print("PyTorch:", torch.__version__)

Python: 3.12.3
Platform: Windows-11-10.0.26200-SP0
NumPy: 2.3.4
Pandas: 2.2.3
XGBoost: 3.1.1
PyTorch: 2.9.1+cpu


In [2]:
# ============================================================
# 1. Global configuration
# ============================================================

RANDOM_STATE = 42

# Keep this False if your X_train/X_test were already cleaned.
# This is set to False by default to reproduce the previous xgboost.ipynb as closely as possible.
# If the raw X files still contain leakage/post-outcome columns, set this to True.
DROP_EXCLUDED_COLS = False

EXCLUDE_COLS = [
    "RecordID",
    "in_hospital_death",
    "SAPS-I",
    "SOFA",
    "Length_of_stay",
    "Survival",
]

TARGET_COL = "in_hospital_death"

# Reproduce the previous xgboost.ipynb AUC model as closely as possible.
RUN_LOGISTIC_REGRESSION = True
RUN_XGB_AUC = True
RUN_XGB_BRIER = True
RUN_MLP = True

# Optional supplementary CSV exports.
# These do not create figures. They only save CSV tables.
RUN_FAST_FEATURE_TABLES = True       # LR coefficients + XGBoost native importance
RUN_OPTIONAL_SHAP_EXPORT = False     # can be slower; requires shap
RUN_OPTIONAL_PERMUTATION = False     # can be slow with many features

# XGBoost settings from the earlier notebook.
XGB_DEVICE = "cuda"    # previous xgboost.ipynb used device="cuda"
XGB_TREE_METHOD = "hist"
N_JOBS_XGB_GRID = 1    # previous notebook used n_jobs=1
N_JOBS_SKLEARN = -1

# Output folders.
BASE_DIR = Path("..")
RESULTS_DIR = BASE_DIR / "output" / "model_training_results"
MODEL_DIR = BASE_DIR / "output" / "trained_models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Results directory:", RESULTS_DIR.resolve())
print("Model directory:", MODEL_DIR.resolve())

Results directory: C:\Users\junse\Documents\research\IUBDC 2026\output\model_training_results
Model directory: C:\Users\junse\Documents\research\IUBDC 2026\output\trained_models


In [3]:
# ============================================================
# 2. Reproducibility helpers
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # More reproducible, but may be slightly slower.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(RANDOM_STATE)

TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch device:", TORCH_DEVICE)

runtime_info = pd.DataFrame([{
    "random_state": RANDOM_STATE,
    "torch_device": str(TORCH_DEVICE),
    "xgb_requested_device": XGB_DEVICE,
    "xgb_tree_method": XGB_TREE_METHOD,
    "drop_excluded_cols": DROP_EXCLUDED_COLS,
}])
runtime_info.to_csv(RESULTS_DIR / "runtime_configuration.csv", index=False)
display(runtime_info)

Torch device: cpu


,random_state,torch_device,xgb_requested_device,xgb_tree_method,drop_excluded_cols
0,42,cpu,cuda,hist,False


In [4]:
# ============================================================
# 3. Locate and load train/test files
# ============================================================

# This notebook is designed for:
# project/
# ├── code/
# │   └── this_notebook.ipynb
# └── output/
#     ├── X_train.csv
#     ├── y_train.csv
#     ├── X_test.csv
#     └── y_test.csv
#
# It also supports the previous path used by xgboost.ipynb:
# ../data/check/IUBDC 2026/preprocessed_dataset/

CANDIDATE_DATA_DIRS = [
    Path("../output"),
    Path("../data/check/IUBDC 2026/preprocessed_dataset"),
    Path("."),
    Path("/mnt/data"),
]

def find_csv_file(exact_name, prefix=None):
    # 1) exact match first
    for directory in CANDIDATE_DATA_DIRS:
        candidate = directory / exact_name
        if candidate.exists():
            return candidate

    # 2) fallback: prefix match, useful for uploaded files like X_train(12).csv
    if prefix is not None:
        for directory in CANDIDATE_DATA_DIRS:
            matches = sorted(directory.glob(f"{prefix}*.csv"))
            if matches:
                return matches[0]

    raise FileNotFoundError(
        f"Could not find {exact_name}. Checked: {[str(d) for d in CANDIDATE_DATA_DIRS]}"
    )

X_TRAIN_PATH = find_csv_file("X_train.csv", prefix="X_train")
Y_TRAIN_PATH = find_csv_file("y_train.csv", prefix="y_train")
X_TEST_PATH  = find_csv_file("X_test.csv", prefix="X_test")
Y_TEST_PATH  = find_csv_file("y_test.csv", prefix="y_test")

print("X_train:", X_TRAIN_PATH)
print("y_train:", Y_TRAIN_PATH)
print("X_test :", X_TEST_PATH)
print("y_test :", Y_TEST_PATH)

X_train_full = pd.read_csv(X_TRAIN_PATH)
y_train_full_df = pd.read_csv(Y_TRAIN_PATH)
X_test = pd.read_csv(X_TEST_PATH)
y_test_df = pd.read_csv(Y_TEST_PATH)

print("Raw X_train shape:", X_train_full.shape)
print("Raw y_train shape:", y_train_full_df.shape)
print("Raw X_test shape :", X_test.shape)
print("Raw y_test shape :", y_test_df.shape)

X_train: ..\output\X_train.csv
y_train: ..\output\y_train.csv
X_test : ..\output\X_test.csv
y_test : ..\output\y_test.csv
Raw X_train shape: (8000, 243)
Raw y_train shape: (8000, 1)
Raw X_test shape : (4000, 243)
Raw y_test shape : (4000, 1)


In [5]:
# ============================================================
# 4. Prepare labels and feature matrix
# ============================================================

def extract_binary_target(y_df, target_col=TARGET_COL):
    if target_col in y_df.columns:
        y = y_df[target_col].values
    elif y_df.shape[1] == 1:
        y = y_df.iloc[:, 0].values
    else:
        raise ValueError(
            f"Could not find target column '{target_col}', and y file has multiple columns: {list(y_df.columns)}"
        )

    y = np.asarray(y).reshape(-1)
    y = pd.Series(y).astype(int).values

    unique_values = sorted(np.unique(y).tolist())
    if unique_values != [0, 1]:
        raise ValueError(f"Expected binary labels 0/1, but found: {unique_values}")

    return y


y_train_full = extract_binary_target(y_train_full_df)
y_test = extract_binary_target(y_test_df)

# Use shared columns only.
common_cols = [c for c in X_train_full.columns if c in X_test.columns]
if len(common_cols) != X_train_full.shape[1] or len(common_cols) != X_test.shape[1]:
    print("[WARN] Train/test columns differ. Using only shared columns.")

X_train_full = X_train_full[common_cols].copy()
X_test = X_test[common_cols].copy()

if DROP_EXCLUDED_COLS:
    drop_cols = [c for c in EXCLUDE_COLS if c in X_train_full.columns]
    if drop_cols:
        print("[INFO] Dropping excluded/leakage columns:", drop_cols)
        X_train_full = X_train_full.drop(columns=drop_cols)
        X_test = X_test.drop(columns=drop_cols)
else:
    present_excluded = [c for c in EXCLUDE_COLS if c in X_train_full.columns]
    if present_excluded:
        print("[WARN] Excluded/leakage-prone columns are present but kept because DROP_EXCLUDED_COLS=False:")
        print(present_excluded)

# Keep numeric columns only. The previous xgboost.ipynb used StandardScaler over all x_train columns.
numeric_cols = X_train_full.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) != X_train_full.shape[1]:
    non_numeric = [c for c in X_train_full.columns if c not in numeric_cols]
    print("[WARN] Non-numeric columns detected and removed:", non_numeric)

X_train_full = X_train_full[numeric_cols].copy()
X_test = X_test[numeric_cols].copy()

feature_names = np.array(X_train_full.columns)
pd.DataFrame({"feature": feature_names}).to_csv(RESULTS_DIR / "feature_names_used.csv", index=False)

cohort_summary = pd.DataFrame([
    {
        "split": "train_full",
        "n_samples": len(y_train_full),
        "n_features": X_train_full.shape[1],
        "death_rate": float(np.mean(y_train_full)),
        "n_survivors": int(np.sum(y_train_full == 0)),
        "n_deaths": int(np.sum(y_train_full == 1)),
    },
    {
        "split": "external_test",
        "n_samples": len(y_test),
        "n_features": X_test.shape[1],
        "death_rate": float(np.mean(y_test)),
        "n_survivors": int(np.sum(y_test == 0)),
        "n_deaths": int(np.sum(y_test == 1)),
    },
])
cohort_summary.to_csv(RESULTS_DIR / "cohort_summary_for_modeling.csv", index=False)
display(cohort_summary)

print("Final X_train_full shape:", X_train_full.shape)
print("Final X_test shape      :", X_test.shape)

,split,n_samples,n_features,death_rate,n_survivors,n_deaths
0,train_full,8000,243,0.14025,6878,1122
1,external_test,4000,243,0.14625,3415,585


Final X_train_full shape: (8000, 243)
Final X_test shape      : (4000, 243)


In [6]:
# ============================================================
# 5. Shared metric and export helpers
# ============================================================

def compute_metrics(y_true, y_prob, threshold=0.50):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_prob = np.asarray(y_prob).reshape(-1)
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "pr_auc": average_precision_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "brier_score": brier_score_loss(y_true, y_prob),
        "threshold": threshold,
    }


def save_classification_report(y_true, y_prob, threshold, model_name):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report).T.reset_index().rename(columns={"index": "class_or_average"})
    report_df.insert(0, "model", model_name)
    report_df.to_csv(RESULTS_DIR / f"{model_name}_classification_report.csv", index=False)

    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["actual_survivor_0", "actual_death_1"],
        columns=["pred_survivor_0", "pred_death_1"],
    )
    cm_df.insert(0, "model", model_name)
    cm_df.to_csv(RESULTS_DIR / f"{model_name}_confusion_matrix.csv")
    return report_df, cm_df


def save_curve_csvs(y_true, y_prob, model_name, n_calibration_bins=10):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_prob = np.asarray(y_prob).reshape(-1)

    fpr, tpr, roc_thresholds = roc_curve(y_true, y_prob)
    roc_df = pd.DataFrame({
        "model": model_name,
        "fpr": fpr,
        "tpr": tpr,
        "threshold": roc_thresholds,
    })
    roc_df.to_csv(RESULTS_DIR / f"{model_name}_roc_curve_points.csv", index=False)

    precision, recall, pr_thresholds = precision_recall_curve(y_true, y_prob)
    pr_thresholds_padded = np.append(pr_thresholds, np.nan)
    pr_df = pd.DataFrame({
        "model": model_name,
        "recall": recall,
        "precision": precision,
        "threshold": pr_thresholds_padded,
        "baseline_death_rate": np.mean(y_true),
    })
    pr_df.to_csv(RESULTS_DIR / f"{model_name}_precision_recall_curve_points.csv", index=False)

    prob_true, prob_pred = calibration_curve(
        y_true,
        y_prob,
        n_bins=n_calibration_bins,
        strategy="quantile",
    )
    cal_df = pd.DataFrame({
        "model": model_name,
        "bin_id": np.arange(len(prob_true)),
        "mean_predicted_probability": prob_pred,
        "observed_event_rate": prob_true,
    })
    cal_df.to_csv(RESULTS_DIR / f"{model_name}_calibration_curve_points.csv", index=False)

    return roc_df, pr_df, cal_df


def flatten_params(params):
    return {k: str(v) for k, v in params.items()}


all_performance_rows = []
external_prediction_df = pd.DataFrame({
    "row_id": np.arange(len(y_test)),
    "true_label": y_test.astype(int),
})
trained_models = {}
model_thresholds = {}
model_probabilities = {}

In [7]:
# ============================================================
# 6. Logistic Regression tuning
# ============================================================

if RUN_LOGISTIC_REGRESSION:
    start = time.time()

    lr_preprocessor = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
        ]
    )

    lr_pipe = Pipeline(
        steps=[
            ("preprocessor", lr_preprocessor),
            ("logisticregression", LogisticRegression(
                solver="liblinear",
                max_iter=5000,
                random_state=RANDOM_STATE,
            )),
        ]
    )

    lr_param_grid = {
        "logisticregression__C": [0.01, 0.1, 1.0, 10.0],
        "logisticregression__penalty": ["l1", "l2"],
        "logisticregression__class_weight": [None, "balanced"],
    }

    lr_grid = GridSearchCV(
        estimator=lr_pipe,
        param_grid=lr_param_grid,
        scoring="roc_auc",
        cv=5,
        n_jobs=N_JOBS_SKLEARN,
        verbose=1,
        return_train_score=True,
    )

    print("Starting Logistic Regression GridSearchCV...")
    lr_grid.fit(X_train_full, y_train_full)

    lr_best = lr_grid.best_estimator_
    lr_prob = lr_best.predict_proba(X_test)[:, 1]
    lr_threshold = 0.50

    lr_cv_results = pd.DataFrame(lr_grid.cv_results_)
    lr_cv_results.to_csv(RESULTS_DIR / "logistic_regression_cv_results.csv", index=False)

    lr_best_params = pd.DataFrame([{
        "model": "logistic_regression",
        "selection_metric": "cv_roc_auc",
        "best_cv_roc_auc": lr_grid.best_score_,
        **flatten_params(lr_grid.best_params_),
    }])
    lr_best_params.to_csv(RESULTS_DIR / "logistic_regression_best_params.csv", index=False)

    metrics = compute_metrics(y_test, lr_prob, threshold=lr_threshold)
    all_performance_rows.append({
        "model": "logistic_regression",
        "model_family": "linear",
        "tuning_objective": "cv_roc_auc",
        "split": "external_test",
        **metrics,
        "fit_seconds": time.time() - start,
    })

    external_prediction_df["logistic_regression_probability"] = lr_prob
    external_prediction_df["logistic_regression_pred_label"] = (lr_prob >= lr_threshold).astype(int)

    save_classification_report(y_test, lr_prob, lr_threshold, "logistic_regression")
    save_curve_csvs(y_test, lr_prob, "logistic_regression")

    trained_models["logistic_regression"] = lr_best
    model_thresholds["logistic_regression"] = lr_threshold
    model_probabilities["logistic_regression"] = lr_prob

    joblib.dump(lr_best, MODEL_DIR / "logistic_regression_best_pipeline.joblib")

    print("Best LR CV ROC-AUC:", round(lr_grid.best_score_, 4))
    print("External LR ROC-AUC:", round(roc_auc_score(y_test, lr_prob), 4))
    print("Saved Logistic Regression CSV outputs.")
else:
    print("Skipping Logistic Regression.")

Starting Logistic Regression GridSearchCV...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best LR CV ROC-AUC: 0.8523
External LR ROC-AUC: 0.8631
Saved Logistic Regression CSV outputs.


In [8]:
# ============================================================
# 7. XGBoost ROC-AUC tuning
#    This reproduces the previous xgboost.ipynb setup as closely as possible.
# ============================================================

# Previous xgboost.ipynb grid:
XGB_PARAM_GRID = {
    "xgbclassifier__max_depth": [4, 6, 8],
    "xgbclassifier__learning_rate": [0.01, 0.05, 0.1],
    "xgbclassifier__n_estimators": [100, 200, 300],
    "xgbclassifier__subsample": [0.8, 1.0],
    "xgbclassifier__colsample_bytree": [0.8, 1.0],
}

# Previous xgboost.ipynb preprocessing:
# numeric_features = x_train.columns
# ColumnTransformer([("numeric", StandardScaler(), numeric_features)])
xgb_numeric_features = X_train_full.columns
xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), xgb_numeric_features)
    ]
)


def build_xgb_auc_pipeline(device_value):
    xgb_base = xgb.XGBClassifier(
        tree_method=XGB_TREE_METHOD,
        device=device_value,
        random_state=RANDOM_STATE,
        eval_metric="auc",
    )
    return make_pipeline(xgb_preprocessor, xgb_base)


def build_xgb_brier_pipeline(device_value):
    xgb_base = xgb.XGBClassifier(
        tree_method=XGB_TREE_METHOD,
        device=device_value,
        random_state=RANDOM_STATE,
        objective="binary:logistic",
    )
    return make_pipeline(xgb_preprocessor, xgb_base)


def fit_xgb_grid_with_fallback(grid_obj, X, y, model_label):
    try:
        grid_obj.fit(X, y)
        return grid_obj, XGB_DEVICE
    except Exception as e:
        if XGB_DEVICE == "cuda":
            print(f"[WARN] {model_label} failed with device='cuda'. Retrying with device='cpu'.")
            print("Original error:", repr(e))

            if model_label == "xgb_auc":
                pipe_cpu = build_xgb_auc_pipeline("cpu")
                scoring = "roc_auc"
            elif model_label == "xgb_brier":
                pipe_cpu = build_xgb_brier_pipeline("cpu")
                scoring = "neg_brier_score"
            else:
                raise ValueError(model_label)

            grid_cpu = GridSearchCV(
                estimator=pipe_cpu,
                param_grid=XGB_PARAM_GRID,
                scoring=scoring,
                cv=5,
                verbose=2,
                n_jobs=N_JOBS_XGB_GRID,
                return_train_score=True,
            )
            grid_cpu.fit(X, y)
            return grid_cpu, "cpu"
        else:
            raise

In [9]:
if RUN_XGB_AUC:
    start = time.time()

    xgb_auc_pipe = build_xgb_auc_pipeline(XGB_DEVICE)

    xgb_auc_grid = GridSearchCV(
        estimator=xgb_auc_pipe,
        param_grid=XGB_PARAM_GRID,
        scoring="roc_auc",   # same as previous xgboost.ipynb
        cv=5,                # same as previous xgboost.ipynb
        verbose=2,
        n_jobs=N_JOBS_XGB_GRID,
        return_train_score=True,
    )

    print("Starting XGBoost ROC-AUC GridSearchCV...")
    xgb_auc_grid, actual_xgb_auc_device = fit_xgb_grid_with_fallback(
        xgb_auc_grid,
        X_train_full,
        y_train_full.ravel(),
        model_label="xgb_auc",
    )

    xgb_auc_best = xgb_auc_grid.best_estimator_
    xgb_auc_prob = xgb_auc_best.predict_proba(X_test)[:, 1]
    xgb_auc_pred = xgb_auc_best.predict(X_test)
    xgb_auc_threshold = 0.50

    xgb_auc_cv_results = pd.DataFrame(xgb_auc_grid.cv_results_)
    xgb_auc_cv_results.to_csv(RESULTS_DIR / "xgboost_auc_cv_results.csv", index=False)

    xgb_auc_best_params = pd.DataFrame([{
        "model": "xgboost_auc_optimized",
        "selection_metric": "cv_roc_auc",
        "best_cv_roc_auc": xgb_auc_grid.best_score_,
        "actual_xgb_device": actual_xgb_auc_device,
        **flatten_params(xgb_auc_grid.best_params_),
    }])
    xgb_auc_best_params.to_csv(RESULTS_DIR / "xgboost_auc_best_params.csv", index=False)

    metrics = compute_metrics(y_test, xgb_auc_prob, threshold=xgb_auc_threshold)
    all_performance_rows.append({
        "model": "xgboost_auc_optimized",
        "model_family": "gradient_boosted_trees",
        "tuning_objective": "cv_roc_auc",
        "split": "external_test",
        **metrics,
        "best_cv_score": xgb_auc_grid.best_score_,
        "actual_xgb_device": actual_xgb_auc_device,
        "fit_seconds": time.time() - start,
    })

    external_prediction_df["xgboost_auc_probability"] = xgb_auc_prob
    external_prediction_df["xgboost_auc_pred_label"] = (xgb_auc_prob >= xgb_auc_threshold).astype(int)

    save_classification_report(y_test, xgb_auc_prob, xgb_auc_threshold, "xgboost_auc_optimized")
    save_curve_csvs(y_test, xgb_auc_prob, "xgboost_auc_optimized")

    trained_models["xgboost_auc_optimized"] = xgb_auc_best
    model_thresholds["xgboost_auc_optimized"] = xgb_auc_threshold
    model_probabilities["xgboost_auc_optimized"] = xgb_auc_prob

    joblib.dump(xgb_auc_best, MODEL_DIR / "xgboost_auc_best_pipeline.joblib")

    print("\n--- XGBoost AUC Grid Search Results ---")
    print(f"Best ROC AUC Score: {xgb_auc_grid.best_score_:.4f}")
    print(f"Best Parameters: {xgb_auc_grid.best_params_}")
    print("\n--- External Test Set Performance ---")
    print(f"Test ROC AUC: {roc_auc_score(y_test, xgb_auc_prob):.4f}")
    print(f"Test Accuracy: {accuracy_score(y_test, xgb_auc_pred):.4f}")
    print(f"Test Brier Score: {brier_score_loss(y_test, xgb_auc_prob):.4f}")
else:
    print("Skipping XGBoost ROC-AUC tuning.")

Starting XGBoost ROC-AUC GridSearchCV...
Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV] END xgbclassifier__colsample_bytree=0.8, xgbclassifier__learning_rate=0.01, xgbclassifier__max_depth=4, xgbclassifier__n_estimators=100, xgbclassifier__subsample=0.8; total time=   0.4s
[CV] END xgbclassifier__colsample_bytree=0.8, xgbclassifier__learning_rate=0.01, xgbclassifier__max_depth=4, xgbclassifier__n_estimators=100, xgbclassifier__subsample=0.8; total time=   0.4s
[CV] END xgbclassifier__colsample_bytree=0.8, xgbclassifier__learning_rate=0.01, xgbclassifier__max_depth=4, xgbclassifier__n_estimators=100, xgbclassifier__subsample=0.8; total time=   0.4s
[CV] END xgbclassifier__colsample_bytree=0.8, xgbclassifier__learning_rate=0.01, xgbclassifier__max_depth=4, xgbclassifier__n_estimators=100, xgbclassifier__subsample=0.8; total time=   0.4s
[CV] END xgbclassifier__colsample_bytree=0.8, xgbclassifier__learning_rate=0.01, xgbclassifier__max_depth=4, xgbclassifier__n_estima

In [10]:
# ============================================================
# 9. MLP preprocessing and internal validation split
# ============================================================

if RUN_MLP:
    # Internal validation split from X_train only.
    X_mlp_train, X_mlp_val, y_mlp_train, y_mlp_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.20,
        stratify=y_train_full,
        random_state=RANDOM_STATE,
    )

    mlp_preprocessor = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
        ]
    )

    X_mlp_train_np = mlp_preprocessor.fit_transform(X_mlp_train).astype(np.float32)
    X_mlp_val_np = mlp_preprocessor.transform(X_mlp_val).astype(np.float32)
    X_mlp_test_np = mlp_preprocessor.transform(X_test).astype(np.float32)

    mlp_feature_names = np.array(X_mlp_train.columns)
    pd.DataFrame({"feature": mlp_feature_names}).to_csv(RESULTS_DIR / "mlp_feature_names.csv", index=False)

    print("MLP internal train:", X_mlp_train_np.shape, "death rate:", y_mlp_train.mean())
    print("MLP internal val  :", X_mlp_val_np.shape, "death rate:", y_mlp_val.mean())
    print("MLP external test :", X_mlp_test_np.shape, "death rate:", y_test.mean())
else:
    print("Skipping MLP setup.")

MLP internal train: (6400, 243) death rate: 0.1403125
MLP internal val  : (1600, 243) death rate: 0.14
MLP external test : (4000, 243) death rate: 0.14625


In [11]:
# ============================================================
# 10. MLP model and helper functions
# ============================================================

if RUN_MLP:
    MAX_EPOCHS = 150
    PATIENCE = 12
    MIN_DELTA = 1e-4
    GRAD_CLIP_NORM = 5.0

    # AUC is primary, but Brier score is lightly penalized because this is risk prediction.
    BRIER_PENALTY = 0.25

    CONFIG_GRID = [
        {"hidden_dims": [64],          "dropout_rate": 0.10, "learning_rate": 1e-3, "weight_decay": 1e-3, "batch_size": 128, "pos_weight_mode": "none"},
        {"hidden_dims": [64],          "dropout_rate": 0.20, "learning_rate": 5e-4, "weight_decay": 1e-3, "batch_size": 128, "pos_weight_mode": "sqrt_balanced"},
        {"hidden_dims": [64, 32],      "dropout_rate": 0.20, "learning_rate": 1e-3, "weight_decay": 1e-3, "batch_size": 128, "pos_weight_mode": "sqrt_balanced"},
        {"hidden_dims": [64, 32],      "dropout_rate": 0.30, "learning_rate": 5e-4, "weight_decay": 1e-3, "batch_size": 128, "pos_weight_mode": "sqrt_balanced"},
        {"hidden_dims": [128, 64],     "dropout_rate": 0.30, "learning_rate": 5e-4, "weight_decay": 1e-3, "batch_size": 128, "pos_weight_mode": "sqrt_balanced"},
        {"hidden_dims": [128, 64],     "dropout_rate": 0.40, "learning_rate": 3e-4, "weight_decay": 1e-2, "batch_size": 128, "pos_weight_mode": "sqrt_balanced"},
        {"hidden_dims": [128, 64, 32], "dropout_rate": 0.40, "learning_rate": 3e-4, "weight_decay": 1e-2, "batch_size": 128, "pos_weight_mode": "sqrt_balanced"},
        {"hidden_dims": [64, 32],      "dropout_rate": 0.30, "learning_rate": 5e-4, "weight_decay": 1e-3, "batch_size": 128, "pos_weight_mode": "balanced"},
    ]

    class FeedForwardMLP(nn.Module):
        def __init__(self, input_dim, hidden_dims, dropout_rate):
            super().__init__()

            layers = []
            prev_dim = input_dim

            for hidden_dim in hidden_dims:
                layers.append(nn.Linear(prev_dim, hidden_dim))
                layers.append(nn.LayerNorm(hidden_dim))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout_rate))
                prev_dim = hidden_dim

            layers.append(nn.Linear(prev_dim, 1))
            self.network = nn.Sequential(*layers)

        def forward(self, x):
            return self.network(x)


    def make_loader(X_np, y, batch_size, shuffle=False, drop_last=False):
        dataset = TensorDataset(
            torch.tensor(X_np, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32).view(-1, 1),
        )
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)


    def get_pos_weight_value(y, mode):
        num_pos = float(np.sum(y))
        num_neg = float(len(y) - num_pos)
        ratio = num_neg / max(num_pos, 1.0)

        if mode == "balanced":
            return ratio
        if mode == "sqrt_balanced":
            return np.sqrt(ratio)
        if mode == "none":
            return 1.0
        raise ValueError(f"Unknown pos_weight_mode: {mode}")


    def run_epoch(model, loader, criterion, optimizer=None):
        is_train = optimizer is not None
        model.train() if is_train else model.eval()

        total_loss = 0.0
        all_probs = []
        all_targets = []

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(TORCH_DEVICE)
            y_batch = y_batch.to(TORCH_DEVICE)

            if is_train:
                optimizer.zero_grad()

            with torch.set_grad_enabled(is_train):
                logits = model(X_batch)
                loss = criterion(logits, y_batch)

                if is_train:
                    loss.backward()
                    if GRAD_CLIP_NORM is not None:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    optimizer.step()

            total_loss += loss.item() * X_batch.size(0)
            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
            targets = y_batch.detach().cpu().numpy().reshape(-1)

            all_probs.append(probs)
            all_targets.append(targets)

        all_probs = np.concatenate(all_probs)
        all_targets = np.concatenate(all_targets)
        avg_loss = total_loss / len(loader.dataset)

        return avg_loss, all_probs, all_targets


    def mlp_selection_score(metrics):
        return metrics["roc_auc"] - BRIER_PENALTY * metrics["brier_score"]


    print("Number of MLP tuning configs:", len(CONFIG_GRID))

Number of MLP tuning configs: 8


In [12]:
# ============================================================
# 11. MLP tuning
# ============================================================

if RUN_MLP:
    def train_one_mlp_config(config, config_id):
        set_seed(RANDOM_STATE + config_id)

        batch_size = config["batch_size"]
        train_loader = make_loader(X_mlp_train_np, y_mlp_train, batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = make_loader(X_mlp_val_np, y_mlp_val, batch_size=batch_size, shuffle=False, drop_last=False)

        model = FeedForwardMLP(
            input_dim=X_mlp_train_np.shape[1],
            hidden_dims=config["hidden_dims"],
            dropout_rate=config["dropout_rate"],
        ).to(TORCH_DEVICE)

        pos_weight_value = get_pos_weight_value(y_mlp_train, config["pos_weight_mode"])
        pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(TORCH_DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config["learning_rate"],
            weight_decay=config["weight_decay"],
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=4,
        )

        best_score = -np.inf
        best_epoch = None
        best_state_dict = None
        patience_counter = 0
        history = []

        for epoch in range(1, MAX_EPOCHS + 1):
            train_loss, train_prob, train_true = run_epoch(model, train_loader, criterion, optimizer=optimizer)
            val_loss, val_prob, val_true = run_epoch(model, val_loader, criterion, optimizer=None)

            train_metrics = compute_metrics(train_true, train_prob, threshold=0.50)
            val_metrics = compute_metrics(val_true, val_prob, threshold=0.50)
            score = mlp_selection_score(val_metrics)
            scheduler.step(score)

            row = {
                "config_id": config_id,
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "selection_score": score,
                "train_roc_auc": train_metrics["roc_auc"],
                "val_roc_auc": val_metrics["roc_auc"],
                "train_pr_auc": train_metrics["pr_auc"],
                "val_pr_auc": val_metrics["pr_auc"],
                "train_brier_score": train_metrics["brier_score"],
                "val_brier_score": val_metrics["brier_score"],
                "train_f1": train_metrics["f1"],
                "val_f1": val_metrics["f1"],
                "learning_rate_now": optimizer.param_groups[0]["lr"],
            }
            history.append(row)

            if score > best_score + MIN_DELTA:
                best_score = score
                best_epoch = epoch
                best_state_dict = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= PATIENCE:
                break

        model.load_state_dict(best_state_dict)

        final_val_loss, final_val_prob, final_val_true = run_epoch(model, val_loader, criterion, optimizer=None)
        final_val_metrics = compute_metrics(final_val_true, final_val_prob, threshold=0.50)
        final_score = mlp_selection_score(final_val_metrics)

        summary = {
            "config_id": config_id,
            "best_epoch": best_epoch,
            "selection_score": final_score,
            "val_loss": final_val_loss,
            **{f"val_{k}": v for k, v in final_val_metrics.items()},
            **config,
            "hidden_dims": str(config["hidden_dims"]),
            "pos_weight_value": pos_weight_value,
        }

        return {
            "model": model,
            "criterion": criterion,
            "history": pd.DataFrame(history),
            "summary": summary,
            "config": config,
            "best_epoch": best_epoch,
            "pos_weight_value": pos_weight_value,
        }


    start = time.time()

    all_mlp_runs = []
    all_mlp_histories = []

    for config_id, config in enumerate(CONFIG_GRID):
        print(f"\nTraining MLP config {config_id + 1}/{len(CONFIG_GRID)}: {config}")
        result = train_one_mlp_config(config, config_id=config_id)
        all_mlp_runs.append(result)
        all_mlp_histories.append(result["history"])

        s = result["summary"]
        print(
            f"best_epoch={s['best_epoch']} | "
            f"val_auc={s['val_roc_auc']:.4f} | "
            f"val_pr_auc={s['val_pr_auc']:.4f} | "
            f"val_brier={s['val_brier_score']:.4f} | "
            f"selection_score={s['selection_score']:.4f}"
        )

    mlp_tuning_results_df = pd.DataFrame([r["summary"] for r in all_mlp_runs])
    mlp_tuning_history_df = pd.concat(all_mlp_histories, ignore_index=True)

    mlp_tuning_results_df = mlp_tuning_results_df.sort_values(
        by=["selection_score", "val_roc_auc", "val_brier_score"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    mlp_tuning_results_df.to_csv(RESULTS_DIR / "mlp_tuning_results.csv", index=False)
    mlp_tuning_history_df.to_csv(RESULTS_DIR / "mlp_tuning_history_all_configs.csv", index=False)

    display(mlp_tuning_results_df.head())
else:
    print("Skipping MLP tuning.")


Training MLP config 1/8: {'hidden_dims': [64], 'dropout_rate': 0.1, 'learning_rate': 0.001, 'weight_decay': 0.001, 'batch_size': 128, 'pos_weight_mode': 'none'}
best_epoch=7 | val_auc=0.8643 | val_pr_auc=0.5450 | val_brier=0.0866 | selection_score=0.8427

Training MLP config 2/8: {'hidden_dims': [64], 'dropout_rate': 0.2, 'learning_rate': 0.0005, 'weight_decay': 0.001, 'batch_size': 128, 'pos_weight_mode': 'sqrt_balanced'}
best_epoch=17 | val_auc=0.8606 | val_pr_auc=0.5345 | val_brier=0.0981 | selection_score=0.8361

Training MLP config 3/8: {'hidden_dims': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'weight_decay': 0.001, 'batch_size': 128, 'pos_weight_mode': 'sqrt_balanced'}
best_epoch=4 | val_auc=0.8626 | val_pr_auc=0.5282 | val_brier=0.1034 | selection_score=0.8368

Training MLP config 4/8: {'hidden_dims': [64, 32], 'dropout_rate': 0.3, 'learning_rate': 0.0005, 'weight_decay': 0.001, 'batch_size': 128, 'pos_weight_mode': 'sqrt_balanced'}
best_epoch=13 | val_auc=0.8656 |

,config_id,best_epoch,selection_score,val_loss,val_roc_auc,val_pr_auc,val_accuracy,val_precision,val_recall,val_f1,val_brier_score,val_threshold,hidden_dims,dropout_rate,learning_rate,weight_decay,batch_size,pos_weight_mode,pos_weight_value
0,0,7,0.842689,0.286297,0.864349,0.544988,0.884375,0.632653,0.415179,0.501348,0.086641,0.5,[64],0.1,0.0010,0.001,128,none,1.000000
1,3,13,0.840626,0.501559,0.865575,0.557422,0.855625,0.487719,0.620536,0.546169,0.099797,0.5,"[64, 32]",0.3,0.0005,0.001,128,sqrt_balanced,2.475267
2,4,6,0.839612,0.502205,0.864650,0.524776,0.852500,0.478571,0.598214,0.531746,0.100155,0.5,"[128, 64]",0.3,0.0005,0.001,128,sqrt_balanced,2.475267
3,2,4,0.836807,0.497866,0.862649,0.528198,0.843750,0.457516,0.625000,0.528302,0.103365,0.5,"[64, 32]",0.2,0.0010,0.001,128,sqrt_balanced,2.475267
4,1,17,0.836062,0.508301,0.860582,0.534467,0.855625,0.486056,0.544643,0.513684,0.098079,0.5,[64],0.2,0.0005,0.001,128,sqrt_balanced,2.475267


In [13]:
# ============================================================
# 12. MLP threshold tuning and final external evaluation
# ============================================================

if RUN_MLP:
    best_mlp_config_id = int(mlp_tuning_results_df.loc[0, "config_id"])
    best_mlp_run = all_mlp_runs[best_mlp_config_id]

    mlp_model = best_mlp_run["model"]
    mlp_criterion = best_mlp_run["criterion"]
    BEST_MLP_CONFIG = best_mlp_run["config"]
    BEST_MLP_EPOCH = best_mlp_run["best_epoch"]
    POS_WEIGHT_VALUE = best_mlp_run["pos_weight_value"]

    val_loader = make_loader(X_mlp_val_np, y_mlp_val, batch_size=BEST_MLP_CONFIG["batch_size"], shuffle=False)
    _, mlp_val_prob, mlp_val_true = run_epoch(mlp_model, val_loader, mlp_criterion, optimizer=None)

    threshold_grid = np.linspace(0.05, 0.95, 181)
    threshold_rows = []

    for threshold in threshold_grid:
        m = compute_metrics(mlp_val_true, mlp_val_prob, threshold=threshold)
        threshold_rows.append({"threshold": threshold, **m})

    mlp_threshold_df = pd.DataFrame(threshold_rows)
    mlp_threshold_df.to_csv(RESULTS_DIR / "mlp_validation_threshold_search.csv", index=False)

    # Practical choice: maximize validation F1. No external test leakage.
    best_threshold_row = mlp_threshold_df.sort_values(
        by=["f1", "recall", "precision"],
        ascending=[False, False, False],
    ).iloc[0]

    MLP_DECISION_THRESHOLD = float(best_threshold_row["threshold"])

    # Final external evaluation.
    mlp_train_loader_eval = make_loader(X_mlp_train_np, y_mlp_train, batch_size=BEST_MLP_CONFIG["batch_size"], shuffle=False)
    mlp_val_loader_eval = make_loader(X_mlp_val_np, y_mlp_val, batch_size=BEST_MLP_CONFIG["batch_size"], shuffle=False)
    mlp_test_loader_eval = make_loader(X_mlp_test_np, y_test, batch_size=BEST_MLP_CONFIG["batch_size"], shuffle=False)

    _, mlp_train_prob, mlp_train_true = run_epoch(mlp_model, mlp_train_loader_eval, mlp_criterion, optimizer=None)
    _, mlp_val_prob, mlp_val_true = run_epoch(mlp_model, mlp_val_loader_eval, mlp_criterion, optimizer=None)
    _, mlp_test_prob, mlp_test_true = run_epoch(mlp_model, mlp_test_loader_eval, mlp_criterion, optimizer=None)

    mlp_internal_performance_df = pd.DataFrame([
        {"model": "mlp", "split": "internal_train_used_for_fit", **compute_metrics(mlp_train_true, mlp_train_prob, threshold=MLP_DECISION_THRESHOLD)},
        {"model": "mlp", "split": "internal_validation_used_for_tuning", **compute_metrics(mlp_val_true, mlp_val_prob, threshold=MLP_DECISION_THRESHOLD)},
        {"model": "mlp", "split": "external_test", **compute_metrics(mlp_test_true, mlp_test_prob, threshold=MLP_DECISION_THRESHOLD)},
    ])
    mlp_internal_performance_df.to_csv(RESULTS_DIR / "mlp_train_val_external_performance.csv", index=False)

    mlp_best_config_df = pd.DataFrame([{
        "model": "mlp",
        "best_config_id": best_mlp_config_id,
        "hidden_dims": str(BEST_MLP_CONFIG["hidden_dims"]),
        "dropout_rate": BEST_MLP_CONFIG["dropout_rate"],
        "learning_rate": BEST_MLP_CONFIG["learning_rate"],
        "weight_decay": BEST_MLP_CONFIG["weight_decay"],
        "batch_size": BEST_MLP_CONFIG["batch_size"],
        "pos_weight_mode": BEST_MLP_CONFIG["pos_weight_mode"],
        "pos_weight_value": float(POS_WEIGHT_VALUE),
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "best_epoch": BEST_MLP_EPOCH,
        "decision_threshold": MLP_DECISION_THRESHOLD,
        "selection_rule": "val_roc_auc - BRIER_PENALTY * val_brier_score",
        "brier_penalty": BRIER_PENALTY,
        "fit_seconds": time.time() - start,
    }])
    mlp_best_config_df.to_csv(RESULTS_DIR / "mlp_best_config_summary.csv", index=False)

    with open(RESULTS_DIR / "mlp_best_config.json", "w") as f:
        json.dump(
            {
                "best_config_id": best_mlp_config_id,
                "best_config": BEST_MLP_CONFIG,
                "best_epoch": int(BEST_MLP_EPOCH),
                "pos_weight_value": float(POS_WEIGHT_VALUE),
                "decision_threshold": float(MLP_DECISION_THRESHOLD),
                "selection_rule": "val_roc_auc - BRIER_PENALTY * val_brier_score",
                "brier_penalty": BRIER_PENALTY,
            },
            f,
            indent=2,
        )

    checkpoint = {
        "model_state_dict": mlp_model.state_dict(),
        "input_dim": X_mlp_train_np.shape[1],
        "best_config": BEST_MLP_CONFIG,
        "best_epoch": BEST_MLP_EPOCH,
        "pos_weight_value": float(POS_WEIGHT_VALUE),
        "decision_threshold": MLP_DECISION_THRESHOLD,
        "feature_names": mlp_feature_names.tolist(),
    }
    torch.save(checkpoint, MODEL_DIR / "mlp_tuned_checkpoint.pt")
    joblib.dump(mlp_preprocessor, MODEL_DIR / "mlp_preprocessor.joblib")

    metrics = compute_metrics(mlp_test_true, mlp_test_prob, threshold=MLP_DECISION_THRESHOLD)
    all_performance_rows.append({
        "model": "mlp",
        "model_family": "feed_forward_neural_network",
        "tuning_objective": "internal_val_auc_minus_brier_penalty",
        "split": "external_test",
        **metrics,
        "best_validation_selection_score": float(mlp_tuning_results_df.loc[0, "selection_score"]),
        "fit_seconds": float(mlp_best_config_df.loc[0, "fit_seconds"]),
    })

    external_prediction_df["mlp_probability"] = mlp_test_prob
    external_prediction_df["mlp_pred_label"] = (mlp_test_prob >= MLP_DECISION_THRESHOLD).astype(int)

    save_classification_report(y_test, mlp_test_prob, MLP_DECISION_THRESHOLD, "mlp")
    save_curve_csvs(y_test, mlp_test_prob, "mlp")

    trained_models["mlp"] = mlp_model
    model_thresholds["mlp"] = MLP_DECISION_THRESHOLD
    model_probabilities["mlp"] = mlp_test_prob

    print("Best MLP config id:", best_mlp_config_id)
    print("Best MLP config:", BEST_MLP_CONFIG)
    print("Best MLP epoch:", BEST_MLP_EPOCH)
    print("Selected MLP threshold:", MLP_DECISION_THRESHOLD)
    print("External MLP ROC-AUC:", round(roc_auc_score(y_test, mlp_test_prob), 4))
else:
    print("Skipping MLP final evaluation.")

Best MLP config id: 0
Best MLP config: {'hidden_dims': [64], 'dropout_rate': 0.1, 'learning_rate': 0.001, 'weight_decay': 0.001, 'batch_size': 128, 'pos_weight_mode': 'none'}
Best MLP epoch: 7
Selected MLP threshold: 0.30999999999999994
External MLP ROC-AUC: 0.8669


In [14]:
# ============================================================
# 13. Save combined performance and prediction CSVs
# ============================================================

combined_performance_df = pd.DataFrame(all_performance_rows)
combined_performance_df = combined_performance_df.sort_values(
    by=["split", "roc_auc"],
    ascending=[True, False],
).reset_index(drop=True)

combined_performance_df.to_csv(RESULTS_DIR / "combined_external_test_performance_summary.csv", index=False)
external_prediction_df.to_csv(RESULTS_DIR / "combined_external_test_predictions.csv", index=False)

# Combined curve data for easier Figure 2 generation later.
combined_roc = []
combined_pr = []
combined_cal = []

for model_name in model_probabilities:
    roc_path = RESULTS_DIR / f"{model_name}_roc_curve_points.csv"
    pr_path = RESULTS_DIR / f"{model_name}_precision_recall_curve_points.csv"
    cal_path = RESULTS_DIR / f"{model_name}_calibration_curve_points.csv"

    if roc_path.exists():
        combined_roc.append(pd.read_csv(roc_path))
    if pr_path.exists():
        combined_pr.append(pd.read_csv(pr_path))
    if cal_path.exists():
        combined_cal.append(pd.read_csv(cal_path))

if combined_roc:
    pd.concat(combined_roc, ignore_index=True).to_csv(RESULTS_DIR / "combined_roc_curve_points.csv", index=False)
if combined_pr:
    pd.concat(combined_pr, ignore_index=True).to_csv(RESULTS_DIR / "combined_precision_recall_curve_points.csv", index=False)
if combined_cal:
    pd.concat(combined_cal, ignore_index=True).to_csv(RESULTS_DIR / "combined_calibration_curve_points.csv", index=False)

# Threshold table.
threshold_df = pd.DataFrame([
    {"model": model_name, "decision_threshold": threshold}
    for model_name, threshold in model_thresholds.items()
])
threshold_df.to_csv(RESULTS_DIR / "combined_decision_thresholds.csv", index=False)

display(combined_performance_df)
print("Saved combined performance:", RESULTS_DIR / "combined_external_test_performance_summary.csv")
print("Saved combined predictions :", RESULTS_DIR / "combined_external_test_predictions.csv")
print("Saved combined ROC/PR/calibration CSVs.")

,model,model_family,tuning_objective,split,roc_auc,pr_auc,accuracy,precision,recall,f1,brier_score,threshold,fit_seconds,best_cv_score,actual_xgb_device,best_validation_selection_score
0,xgboost_auc_optimized,gradient_boosted_trees,cv_roc_auc,external_test,0.875175,0.584445,0.88050,0.690391,0.331624,0.448037,0.086970,0.50,1196.497683,0.860183,cuda,NaN
1,mlp,feed_forward_neural_network,internal_val_auc_minus_brier_penalty,external_test,0.866851,0.546140,0.85075,0.491549,0.596581,0.538996,0.090938,0.31,57.604204,NaN,NaN,0.842689
2,logistic_regression,linear,cv_roc_auc,external_test,0.863112,0.548279,0.78125,0.376701,0.757265,0.503123,0.149773,0.50,116.173948,NaN,NaN,NaN


Saved combined performance: ..\output\model_training_results\combined_external_test_performance_summary.csv
Saved combined predictions : ..\output\model_training_results\combined_external_test_predictions.csv
Saved combined ROC/PR/calibration CSVs.


In [15]:
# ============================================================
# 14. Fast feature-importance CSV exports
#     No plots are created here.
# ============================================================

if RUN_FAST_FEATURE_TABLES:
    # Logistic Regression standardized coefficients.
    if "logistic_regression" in trained_models:
        lr_best = trained_models["logistic_regression"]
        lr_coef = lr_best.named_steps["logisticregression"].coef_.reshape(-1)

        lr_coef_df = pd.DataFrame({
            "feature": feature_names,
            "coefficient": lr_coef,
            "abs_coefficient": np.abs(lr_coef),
        }).sort_values("abs_coefficient", ascending=False)

        lr_coef_df.to_csv(RESULTS_DIR / "logistic_regression_standardized_coefficients.csv", index=False)
        print("Saved Logistic Regression coefficients.")

    # XGBoost native feature importance for AUC-optimized and Brier-optimized models.
    for xgb_model_name in ["xgboost_auc_optimized", "xgboost_brier_optimized"]:
        if xgb_model_name in trained_models:
            xgb_pipe = trained_models[xgb_model_name]
            booster_model = xgb_pipe.named_steps["xgbclassifier"]
            try:
                transformed_feature_names = xgb_pipe.named_steps["columntransformer"].get_feature_names_out()
                transformed_feature_names = [name.replace("numeric__", "") for name in transformed_feature_names]
            except Exception:
                transformed_feature_names = list(feature_names)

            importance_df = pd.DataFrame({
                "feature": transformed_feature_names,
                "xgb_feature_importance": booster_model.feature_importances_,
            }).sort_values("xgb_feature_importance", ascending=False)

            importance_df.insert(0, "model", xgb_model_name)
            importance_df.to_csv(RESULTS_DIR / f"{xgb_model_name}_native_feature_importance.csv", index=False)
            print(f"Saved native XGBoost importance: {xgb_model_name}")

else:
    print("Skipping fast feature-importance CSV exports.")

Saved Logistic Regression coefficients.
Saved native XGBoost importance: xgboost_auc_optimized


In [16]:
# ============================================================
# 15. Optional SHAP CSV export for XGBoost
#     This is optional and can be slower.
#     It saves mean absolute SHAP values as CSV only.
# ============================================================

if RUN_OPTIONAL_SHAP_EXPORT:
    try:
        import shap

        SHAP_SAMPLE_SIZE = min(1000, len(X_test))
        shap_sample = X_test.sample(n=SHAP_SAMPLE_SIZE, random_state=RANDOM_STATE)

        for xgb_model_name in ["xgboost_auc_optimized", "xgboost_brier_optimized"]:
            if xgb_model_name not in trained_models:
                continue

            xgb_pipe = trained_models[xgb_model_name]
            xgb_model = xgb_pipe.named_steps["xgbclassifier"]
            xgb_pre = xgb_pipe.named_steps["columntransformer"]

            X_shap_transformed = xgb_pre.transform(shap_sample)

            try:
                shap_feature_names = xgb_pre.get_feature_names_out()
                shap_feature_names = [name.replace("numeric__", "") for name in shap_feature_names]
            except Exception:
                shap_feature_names = list(feature_names)

            explainer = shap.TreeExplainer(xgb_model)
            shap_values = explainer.shap_values(X_shap_transformed)

            # For binary classification, SHAP may return a list depending on version.
            if isinstance(shap_values, list):
                shap_values_arr = shap_values[-1]
            else:
                shap_values_arr = shap_values

            shap_importance_df = pd.DataFrame({
                "feature": shap_feature_names,
                "mean_abs_shap": np.abs(shap_values_arr).mean(axis=0),
            }).sort_values("mean_abs_shap", ascending=False)

            shap_importance_df.insert(0, "model", xgb_model_name)
            shap_importance_df.to_csv(RESULTS_DIR / f"{xgb_model_name}_mean_abs_shap.csv", index=False)
            print(f"Saved SHAP mean absolute values: {xgb_model_name}")

    except ImportError:
        print("[WARN] shap is not installed. Skipping SHAP export.")
else:
    print("RUN_OPTIONAL_SHAP_EXPORT=False, skipping SHAP CSV export.")

RUN_OPTIONAL_SHAP_EXPORT=False, skipping SHAP CSV export.


In [17]:
# ============================================================
# 16. Optional permutation-importance CSV export
#     This is optional and can be slow with many features.
#     It uses external test data for post-hoc reporting only.
# ============================================================

if RUN_OPTIONAL_PERMUTATION:
    PERMUTATION_REPEATS = 5

    # Sklearn models: Logistic Regression and XGBoost pipelines.
    for model_name in ["logistic_regression", "xgboost_auc_optimized", "xgboost_brier_optimized"]:
        if model_name not in trained_models:
            continue

        print(f"Running permutation importance for {model_name}...")
        estimator = trained_models[model_name]

        perm = permutation_importance(
            estimator,
            X_test,
            y_test,
            scoring="roc_auc",
            n_repeats=PERMUTATION_REPEATS,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS_SKLEARN,
        )

        perm_df = pd.DataFrame({
            "model": model_name,
            "feature": feature_names,
            "importance_mean_auc_drop": perm.importances_mean,
            "importance_std": perm.importances_std,
        }).sort_values("importance_mean_auc_drop", ascending=False)

        perm_df.to_csv(RESULTS_DIR / f"{model_name}_permutation_importance_auc_drop.csv", index=False)

    # MLP custom permutation importance.
    if "mlp" in trained_models:
        print("Running permutation importance for MLP...")
        baseline_auc = roc_auc_score(y_test, model_probabilities["mlp"])
        rows = []
        rng = np.random.default_rng(RANDOM_STATE)

        def predict_mlp_proba_from_dataframe(X_df):
            X_np = mlp_preprocessor.transform(X_df).astype(np.float32)
            loader = make_loader(X_np, y_test, batch_size=BEST_MLP_CONFIG["batch_size"], shuffle=False)
            _, prob, _ = run_epoch(mlp_model, loader, mlp_criterion, optimizer=None)
            return prob

        for feature in feature_names:
            drops = []
            for repeat in range(PERMUTATION_REPEATS):
                X_perm = X_test.copy()
                X_perm[feature] = rng.permutation(X_perm[feature].values)
                prob_perm = predict_mlp_proba_from_dataframe(X_perm)
                perm_auc = roc_auc_score(y_test, prob_perm)
                drops.append(baseline_auc - perm_auc)

            rows.append({
                "model": "mlp",
                "feature": feature,
                "importance_mean_auc_drop": float(np.mean(drops)),
                "importance_std": float(np.std(drops)),
            })

        mlp_perm_df = pd.DataFrame(rows).sort_values("importance_mean_auc_drop", ascending=False)
        mlp_perm_df.to_csv(RESULTS_DIR / "mlp_permutation_importance_auc_drop.csv", index=False)

    print("Saved optional permutation-importance CSVs.")
else:
    print("RUN_OPTIONAL_PERMUTATION=False, skipping permutation-importance CSV export.")

RUN_OPTIONAL_PERMUTATION=False, skipping permutation-importance CSV export.


In [18]:
# ============================================================
# 17. Final output manifest
# ============================================================

manifest_rows = []
for path in sorted(RESULTS_DIR.glob("*")):
    if path.is_file():
        manifest_rows.append({
            "file": path.name,
            "relative_path": str(path),
            "size_bytes": path.stat().st_size,
        })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(RESULTS_DIR / "output_manifest.csv", index=False)

display(manifest_df)

print("\nDone. Main CSV files to use next:")
print("1) combined_external_test_performance_summary.csv")
print("2) combined_external_test_predictions.csv")
print("3) combined_roc_curve_points.csv")
print("4) combined_precision_recall_curve_points.csv")
print("5) combined_calibration_curve_points.csv")
print("6) xgboost_auc_cv_results.csv")
print("7) mlp_tuning_results.csv")

,file,relative_path,size_bytes
0,cohort_summary_for_modeling.csv,..\output\model_training_results\cohort_summar...,140
1,combined_calibration_curve_points.csv,..\output\model_training_results\combined_cali...,1364
2,combined_decision_thresholds.csv,..\output\model_training_results\combined_deci...,103
3,combined_external_test_performance_summary.csv,..\output\model_training_results\combined_exte...,894
4,combined_external_test_predictions.csv,..\output\model_training_results\combined_exte...,224300
5,combined_precision_recall_curve_points.csv,..\output\model_training_results\combined_prec...,959203
6,combined_roc_curve_points.csv,..\output\model_training_results\combined_roc_...,146351
7,feature_names_used.csv,..\output\model_training_results\feature_names...,3034
8,logistic_regression_best_params.csv,..\output\model_training_results\logistic_regr...,190
9,logistic_regression_calibration_curve_points.csv,..\output\model_training_results\logistic_regr...,546



Done. Main CSV files to use next:
1) combined_external_test_performance_summary.csv
2) combined_external_test_predictions.csv
3) combined_roc_curve_points.csv
4) combined_precision_recall_curve_points.csv
5) combined_calibration_curve_points.csv
6) xgboost_auc_cv_results.csv
7) mlp_tuning_results.csv
